In [4]:
import os
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

C:\Users\M16\AppData\Local\Temp\ipykernel_19728\3933654057.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


on charge le pdp, on enrichie leurs metadata , on regroupe leur contenu dans all_documents 

In [5]:
def process_all_pdf(pdf_directory):
    """process all pdef files in a directory"""
    all_documents=[]
    pdf_dir=Path(pdf_directory)
    pdf_files=list(pdf_dir.glob("**/*.pdf"))
    for pdf_file in pdf_files:
        print(f"processing: {pdf_file.name}")
        try:
            loader=PyPDFLoader(str(pdf_file))#extraire le contenu du pdf
            documents=loader.load()#retourne la liste des doc
            for doc in documents: #liste des doc extrait d un seul fichier
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type'] = 'pdf'
            all_documents.extend(documents)
            print(f"loaded{len(documents)}pages")
        except Exception as e:
            print(f"error:{e}")
    print(f"total documents loaded:{len(all_documents)}")
    return all_documents

all_pdf_documents= process_all_pdf("../Data")

processing: embeddings_for_rag.pdf
loaded2pages
processing: object_detection_basics.pdf
loaded2pages
total documents loaded:4


chunking


In [6]:
def split_documents(documents,chunk_size=1000, chunk_overlap=200):
    """split documetns into smaller chunks for better RAG performence"""
    text_splitter=RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n", "\n"," ",""]
    ) #create the splitter
    split_docs=text_splitter.split_documents(documents)#split the doc
    print(f"split{len(documents)}documents into{len(split_docs)}chunks")

    if split_docs:#not empty
        print(f"\nExample chunk:")
        print(f"Content: {split_docs[0].page_content[:200]} ... ")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [7]:
chunks=split_documents(all_pdf_documents)

split4documents into8chunks

Example chunk:
Content: RAG learning sample  For practice and testing
Page 1
Embeddings: Turning Text into Meaningful
Numbers
A short learning document for experimenting with a Retrieval-Augmented Generation (RAG) pipeline. ... 
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-15T18:59:59+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-15T18:59:59+00:00', 'subject': '(unspecified)', 'title': 'Embeddings for RAG', 'trapped': '/False', 'source': '..\\Data\\pdf\\embeddings_for_rag.pdf', 'total_pages': 2, 'page': 0, 'page_label': '1', 'source_file': 'embeddings_for_rag.pdf', 'file_type': 'pdf'}


In [8]:
from typing import List
import numpy as np
    
import uuid
from typing import Any, List

import numpy as np
from langchain_core.documents import Document


convert the text into vectorsss

In [10]:
from sentence_transformers import SentenceTransformer


class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):#constructor
        """
        Initialize the embedding manager.

        Args:
            model_name: HuggingFace model name for sentence embeddings.
        """
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        """Load the SentenceTransformer model."""
        try:
            print(f"Loading embedding model: {self.model_name}")

            self.model = SentenceTransformer(self.model_name)#hiya li taatina encode()

            print(
                "Model loaded successfully. "
                f"Embedding dimension: "
                f"{self.model.get_sentence_embedding_dimension()}"
            )

        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generate embeddings for a list of texts.

        Args:
            texts: List of text strings to embed.

        Returns:
            NumPy array of embeddings with shape
            (len(texts), embedding_dim).
        """
        if self.model is None:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts...")

        embeddings = self.model.encode(
            texts,
            show_progress_bar=True
        )

        print(f"Generated embeddings with shape: {embeddings.shape}")

        return embeddings
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2652.59it/s]


Model loaded successfully. Embedding dimension: 384


C:\Users\M16\AppData\Local\Temp\ipykernel_19728\885263596.py:28: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  f"{self.model.get_sentence_embedding_dimension()}"


stocker embeddings dans chromabd

In [11]:
import os
import chromadb


class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(
        self,
        collection_name: str = "pdf_documents",
        persist_directory: str = "./data/vector_store"
    ):
        """
        Initialize the vector store.

        Args:
            collection_name: Name of the ChromaDB collection.
            persist_directory: Directory to persist the vector store.
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None

        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromaDB client and collection."""
        try:
            # Create the directory if it doesn't exist
            os.makedirs(self.persist_directory, exist_ok=True)

            # Create a persistent ChromaDB client sur disque
            self.client = chromadb.PersistentClient(
                path=self.persist_directory
            )

            # Get or create the collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={
                    "description": "PDF document embeddings for RAG"
                }
            )

            print(
                f"Vector store initialized. "
                f"Collection: {self.collection_name}"
            )
            print(
                f"Existing documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise


    def add_documents(self, documents: List[Document], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store.

        Args:
            documents: List of LangChain documents.
            embeddings: Corresponding embeddings for the documents.
        """

        # 1. Vérifier que le nombre de documents correspond
        # au nombre d'embeddings
        if len(documents) != len(embeddings):
            raise ValueError(
                "Number of documents must match number of embeddings"
            )

        print(f"Adding {len(documents)} documents to vector store...")

        # 2. Préparer les données pour ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []

        # 3. Parcourir chaque document et son embedding
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):

            # Générer un identifiant unique
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Préparer les métadonnées
            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            metadatas.append(metadata)

            # Récupérer le contenu textuel
            documents_text.append(doc.page_content)

            # Convertir l'embedding NumPy en liste Python
            embeddings_list.append(embedding.tolist())

        # 4. Ajouter les données à ChromaDB
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )

            print(
                f"Successfully added {len(documents)} documents "
                "to vector store"
            )
            print(
                f"Total documents in collection: "
                f"{self.collection.count()}"
            )

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise
vectorstore=VectorStore()
vectorstore

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 24


In [12]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store in the vector dtaabase
vectorstore. add_documents(chunks, embeddings)

Generating embeddings for 8 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.58it/s]

Generated embeddings with shape: (8, 384)
Adding 8 documents to vector store...
Successfully added 8 documents to vector store
Total documents in collection: 32


retriver pipeline from vectorstore

In [16]:
from typing import List, Dict, Any
import numpy as np

class RAGRetriever:
    """Handles query-based retrieval from the vector store."""

    def __init__(
        self,
        vector_store: VectorStore,
        embedding_manager: EmbeddingManager
    ):
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(
        self,
        query: str,
        top_k: int = 5,
        score_threshold: float = 0.0
    ) -> List[Dict[str, Any]]:

        print(f"Retrieving documents for query: '{query}'")
        print(f"Top k: {top_k}, score threshold: {score_threshold}")

        # Generate the embedding of the query
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            retrieved_docs = []

            if results["documents"] and results["documents"][0]:
                documents = results["documents"][0]
                metadatas = results["metadatas"][0]
                distances = results["distances"][0]
                ids = results["ids"][0]

                for i, (doc_id, document, metadata, distance) in enumerate(
                    zip(ids, documents, metadatas, distances)
                ):
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            "id": doc_id,
                            "content": document,
                            "metadata": metadata,
                            "similarity_score": similarity_score,
                            "rank": i + 1
                        })

            print(f"Retrieved {len(retrieved_docs)} documents")
            return retrieved_docs

        except Exception as e:
            print(f"Error retrieving documents: {e}")
            return []

rag_retriever=RAGRetriever(vectorstore, embedding_manager)

In [17]:
rag_retriever

In [21]:
rag_retriever.retrieve("Why does RAG use embeddings?")

Retrieving documents for query: 'Why does RAG use embeddings?'
Top k: 5, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 28.00it/s]

Generated embeddings with shape: (1, 384)
Retrieved 5 documents


[{'id': 'doc_486e2b93_0',
  'content': 'RAG learning sample \x7f For practice and testing\nPage 1\nEmbeddings: Turning Text into Meaningful\nNumbers\nA short learning document for experimenting with a Retrieval-Augmented Generation (RAG) pipeline.\n1. What is an embedding?\nAn embedding is a list of numbers (a vector) produced by a machine-learning model from data such as a\nsentence, paragraph, image, or audio clip. For text, the vector is designed to represent useful semantic\nfeatures of the text. Texts with related meanings often have vectors that are close to each other in the\nembedding space.\n2. Why does RAG use embeddings?\nA RAG system first extracts text from documents and splits it into smaller pieces called chunks. An\nembedding model converts each chunk into a vector. The vectors and their associated text are stored in a\nvector database such as ChromaDB. When a user asks a question, the system embeds the question and\nsearches for chunks with similar vectors. The retriev

integration vectordb context pipeline with llm output

In [23]:
###simple rag pipeline
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv
load_dotenv()
llm=ChatGroq(model="qwen/qwen3.8-27b",temperature=0.1, max_tokens=1024)



In [26]:
## simple rag function retirevce context+ generate response
def rag_simple(query, retriever,llm,top_k=3):
    results=retriever.retrieve(query,top_k=top_k)
    context="\n\n".join([doc['content']for doc in results])if results else ""
    if not context:
        return "no relevent context"
    prompt=f"""use the following context to answer the qst concisely
    context:{context}
    question:{query}
    answer:
"""
    response=llm.invoke([prompt.format(context=context,query=query)])
    return response.content

In [27]:
answer=rag_simple("Why does RAG use embeddings?",rag_retriever,llm)

Retrieving documents for query: 'Why does RAG use embeddings?'
Top k: 3, score threshold: 0.0
Generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.72it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents


In [28]:
print(answer)

RAG uses embeddings to convert text chunks into vectors stored in a vector database, allowing the system to search for and retrieve semantically similar content to provide as context to a language model when answering user questions.
